# **OPTIMIZACIÓN DE HIPERPARÁMETROS: Random Search + K-Fold Cross-Validation**

Este notebook implementa un proceso completo de **Fine-Tuning** del Deep Ensemble Learning (LSTM + CNN-1D + BiGRU) combinando:

1. **Random Search**: Muestreo aleatorio del espacio de hiperparámetros.
2. **K-Fold Cross-Validation con GroupKFold (K=5)**: Cada configuración se evalúa sobre 5 folds, respetando la integridad de las series temporales por `ID_Serie`.
3. **Optimización de Redes**: Pruning (poda) y Quantization-Aware Training (QAT) para modelos más ligeros en producción.

Los resultados de **todas** las pruebas se exportan a CSV, y los hiperparámetros óptimos se exportan a JSON.

### **1. INSTALACIÓN DE DEPENDENCIAS**

In [1]:
import os
import tensorflow as tf

# Vinculamos la ruta de las librerías matemáticas de sistema
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-12.3/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

print(f"TensorFlow versión: {tf.__version__}")
print(f"GPU disponible: {len(tf.config.list_physical_devices('GPU')) > 0}")

TensorFlow versión: 2.21.0
GPU disponible: False


In [2]:
import sys
!{sys.executable} -m pip install --upgrade pandas pyarrow --quiet
print("✅ Pandas y PyArrow actualizados a sus versiones compatibles.")

✅ Pandas y PyArrow actualizados a sus versiones compatibles.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### **2. LIBRERÍAS Y CONFIGURACIÓN GLOBAL**

In [4]:
# ==============================================================================
# 2. IMPORTACIÓN DE LIBRERÍAS Y CONFIGURACIÓN INICIAL
# ==============================================================================

# --- Manipulación de Datos y Visualización ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- Utilidades del Sistema ---
import os
import time
import gc
import random
import json
from pathlib import Path
from tqdm import tqdm

# --- Deep Learning: TensorFlow y Keras ---
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import (
    Input, Dense, Dropout, BatchNormalization,
    LSTM, GRU, Bidirectional,
    Conv1D, GlobalAveragePooling1D
)

# --- Scikit-Learn ---
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, train_test_split

# NOTA: tensorflow_model_optimization (tfmot) se importa más adelante, solo en las secciones 11 y 12 donde realmente se necesita.

# --- Configuración Global ---
pd.set_option('display.max_columns', None)
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

print("✅ Librerías cargadas correctamente.")
print(f"   TensorFlow: {tf.__version__}")
print(f"   GPU disponible: {len(tf.config.list_physical_devices('GPU')) > 0}")

✅ Librerías cargadas correctamente.
   TensorFlow: 2.21.0
   GPU disponible: False


In [5]:
NUM_HILOS = 32

# Hilos para operaciones matemáticas independientes (multiplicación de matrices)
tf.config.threading.set_intra_op_parallelism_threads(NUM_HILOS)

# Hilos para operaciones que se pueden ejecutar a la vez
tf.config.threading.set_inter_op_parallelism_threads(NUM_HILOS)

### **3. CONFIGURACIÓN DE RUTAS E HIPERPARÁMETROS DEL PROCESO**

In [6]:
# ==============================================================================
# 3. CONFIGURACIÓN
# ==============================================================================

# RUTAS
INPUT_DIR = Path("../../data/processed/")
NOMBRE_INPUT = "data_vin_processed.parquet"

TUNING_OUTPUT_DIR = Path("./tuning_dir/")
TUNING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# PARÁMETROS DEL PROCESO DE TUNING
DIAS_CONTEXTO  = 7
WINDOW_SIZE    = DIAS_CONTEXTO * 24      # Horas de contexto 
N_FOLDS        = 5       # Número de folds para Cross-Validation (GroupKFold)
N_TRIALS       = 10      # Combinaciones aleatorias a probar por cada arquitectura
EPOCHS_TUNE    = 10      # Épocas máximas por entrenamiento de tuning
EARLY_STOPPING = 2       # Patience del EarlyStopping

print(f"📋 Configuración del Tuning:")
print(f"   Ventana temporal:  {WINDOW_SIZE}h")
print(f"   K-Folds:           {N_FOLDS}")
print(f"   Trials (Random):   {N_TRIALS} por modelo")
print(f"   Épocas por trial:  {EPOCHS_TUNE}")
print(f"   Early Stopping:    patience={EARLY_STOPPING}")
print(f"   Directorio salida: {TUNING_OUTPUT_DIR.resolve()}")

📋 Configuración del Tuning:
   Ventana temporal:  168h
   K-Folds:           5
   Trials (Random):   10 por modelo
   Épocas por trial:  10
   Early Stopping:    patience=2
   Directorio salida: C:\Users\User\Desktop\IA-pAir-RETECH-PAN\modelo_3_enfermedades_plagas\notebooks\tuning_hyperparameters\tuning_dir


### **4. CARGA DE DATOS Y PREPROCESAMIENTO**

Se sigue **exactamente** la misma lógica del notebook principal `CrossValidation_Tuning.ipynb`:
- Reindexación alfabética de columnas.
- Feature Engineering: `Hora_Sin`, `Hora_Cos`.
- Codificación de targets con `LabelEncoder`.

> **⚠️ NOTA**: Los datos **NO** se escalan aquí. El escalado se realiza **por fold** dentro del bucle de Cross-Validation para evitar data leakage.

In [7]:
# ==============================================================================
# 4. CARGA Y PREPROCESAMIENTO (IDÉNTICO AL NOTEBOOK PRINCIPAL)
# ==============================================================================

ruta_raw = INPUT_DIR / NOMBRE_INPUT
print("📂 Cargando datos desde:", ruta_raw.resolve())
df = pd.read_parquet(ruta_raw)

# 0. Asegurar formato
df = df.reindex(sorted(df.columns), axis=1)
cols_str = ['Etiqueta_Clase', 'Clase_Entrenamiento']
for col in cols_str:
    if col in df.columns:
        df[col] = df[col].astype(str)

# 2. Codificación de Targets
le = LabelEncoder()
df['Etiqueta_Num'] = le.fit_transform(df['Clase_Entrenamiento'])
print(f"✅ Clases codificadas: {dict(zip(le.classes_, range(len(le.classes_))))}")

# 3. Selección de Variables
cols_features = [
    'Temp_Amb_C', 
    'Hum_Rel_Pct', 'Lluvia_mm', 'Viento_kmh', 
    'Horas_Humedad_Foliar', 'GDD_Acumulado', 
    'Hum_Suelo_Pct', 'pH_Suelo',
    'Hora_Sin', 'Hora_Cos',
    'CO2_ppm', 'VOC_ppb'
]

target_class = 'Etiqueta_Num'
target_reg   = 'Grado_Infeccion'

NUM_FEATURES = len(cols_features)
NUM_CLASSES  = len(le.classes_)

# 4. SPLIT ESTRATÉGICO ESTRATIFICADO (POR ID_SERIE, NUNCA ALEATORIO POR FILA)
# 1. Obtenemos un diccionario 1 a 1 de cada ID_Serie con su patología general
df_unicos = df.drop_duplicates(subset=['ID_Serie'])[['ID_Serie', 'Etiqueta_Clase']]

ids = df_unicos['ID_Serie']
etiquetas = df_unicos['Etiqueta_Clase']

# 2. Primer Split: 70% Train, 30% Temp (Estratificado por la etiqueta)
train_ids, temp_ids, _, temp_etiquetas = train_test_split(
    ids, etiquetas, 
    test_size=0.30, 
    random_state=42, 
    stratify=etiquetas
)

# 3. Segundo Split: Dividimos el 30% restante en 15% Val y 15% Test
val_ids, test_ids = train_test_split(
    temp_ids, 
    test_size=0.50, 
    random_state=42, 
    stratify=temp_etiquetas
)

# 4. Reconstruimos los DataFrames completos usando los IDs seleccionados
train_df = df[df['ID_Serie'].isin(train_ids)].copy()
val_df = df[df['ID_Serie'].isin(val_ids)].copy()
test_df = df[df['ID_Serie'].isin(test_ids)].copy()

print(f"📊 Features: {NUM_FEATURES} | Clases: {NUM_CLASSES}")
print(f"📊 Total registros originales: {len(df):,}")
print(f"📊 Series separadas -> Train: {len(train_ids)} | Val: {len(val_ids)} | Test: {len(test_ids)}")

📂 Cargando datos desde: C:\Users\User\Desktop\IA-pAir-RETECH-PAN\modelo_3_enfermedades_plagas\data\processed\data_vin_processed.parquet
✅ Clases codificadas: {'ALTICA': 0, 'BLACK_ROT': 1, 'BOTRYTIS': 2, 'EMPOASCA': 3, 'ERINOSIS': 4, 'ESCA': 5, 'HEALTHY': 6, 'LOBESIA': 7, 'MILDIU': 8, 'OIDIO': 9, 'RED_MITE': 10}
📊 Features: 12 | Clases: 11
📊 Total registros originales: 1,628,661
📊 Series separadas -> Train: 692 | Val: 148 | Test: 149


### **5. GENERACIÓN DE SECUENCIAS TEMPORALES CON TRACKING DE GRUPOS**

Se generan las ventanas sobre **todo** el dataset (SIN escalar), conservando un vector `groups` que asocia cada ventana con su `ID_Serie`. Esto es esencial para que `GroupKFold` nunca mezcle datos de una misma planta entre train y validation.

In [8]:
# ==============================================================================
# 5. GENERACIÓN DE VENTANAS TEMPORALES CON GRUPOS (VERSIÓN OPTIMIZADA EN RAM)
# ==============================================================================

def crear_secuencias_con_grupos_optimo(df_full, window_size, features, t_class, t_reg):
    """
    Genera ventanas de tamaño `window_size` pre-localizando la memoria RAM 
    para evitar el colapso (Out of Memory) del sistema.
    """
    grupos_df = list(df_full.groupby('ID_Serie'))
    
    # 1. Calcular el número exacto de ventanas que vamos a generar
    total_ventanas = sum(max(0, len(grupo) - window_size) for _, grupo in grupos_df)
    print(f"📐 Reservando RAM para {total_ventanas:,} secuencias...")
    
    # 2. Pre-localizar memoria en NumPy (La clave para evitar el crash)
    X = np.zeros((total_ventanas, window_size, len(features)), dtype=np.float32)
    y_c = np.zeros(total_ventanas, dtype=np.int8)
    y_r = np.zeros(total_ventanas, dtype=np.float32)
    groups = np.zeros(total_ventanas, dtype=np.int32)
    
    # 3. Rellenar las matrices por índice (sin usar listas ni .append)
    idx_actual = 0
    
    for id_serie, grupo in tqdm(grupos_df, desc="Generando tensores 3D"):
        data_features = grupo[features].values
        data_class    = grupo[t_class].values
        data_reg      = grupo[t_reg].values
        
        n_filas = len(data_features)
        n_ventanas_grupo = max(0, n_filas - window_size)
        
        for i in range(n_ventanas_grupo):
            X[idx_actual] = data_features[i : i + window_size]
            y_c[idx_actual] = data_class[i + window_size]
            y_r[idx_actual] = data_reg[i + window_size]
            groups[idx_actual] = id_serie
            
            idx_actual += 1
            
    return X, y_c, y_r, groups

print("\n📦 Empaquetando datos de Train en tensores 3D...")
X_train, y_train_c, y_train_r, groups_train = crear_secuencias_con_grupos_optimo(
    train_df, WINDOW_SIZE, cols_features, target_class, target_reg
)

print("\n📦 Empaquetando datos de Val en tensores 3D...")
X_val, y_val_c, y_val_r, groups_val = crear_secuencias_con_grupos_optimo(
    val_df, WINDOW_SIZE, cols_features, target_class, target_reg
)

# Combinamos Train y Val para el Cross-Validation 
X_all = np.concatenate([X_train, X_val], axis=0)
y_all_c = np.concatenate([y_train_c, y_val_c], axis=0)
y_all_r = np.concatenate([y_train_r, y_val_r], axis=0)
groups_all = np.concatenate([groups_train, groups_val], axis=0)

# Liberar SOLO los DataFrames originales, dejamos las matrices vivas 
# porque la Celda 8 las necesita para el Tuning inicial.
del df, train_df, val_df, test_df
gc.collect()

print(f"\n✅ Secuencias generadas (Train + Val):")
print(f"   X shape:      {X_all.shape}")
print(f"   y_class shape: {y_all_c.shape}")
print(f"   y_reg shape:   {y_all_r.shape}")
print(f"   groups shape:  {groups_all.shape}")
print(f"   Grupos únicos: {len(np.unique(groups_all))}")


📦 Empaquetando datos de Train en tensores 3D...
📐 Reservando RAM para 1,029,109 secuencias...


Generando tensores 3D: 100%|██████████| 692/692 [00:02<00:00, 309.52it/s]



📦 Empaquetando datos de Val en tensores 3D...
📐 Reservando RAM para 209,396 secuencias...


Generando tensores 3D: 100%|██████████| 148/148 [00:00<00:00, 310.53it/s]



✅ Secuencias generadas (Train + Val):
   X shape:      (1238505, 168, 12)
   y_class shape: (1238505,)
   y_reg shape:   (1238505,)
   groups shape:  (1238505,)
   Grupos únicos: 831


### **6. DEFINICIÓN DEL ESPACIO DE BÚSQUEDA**

Se define un diccionario con los rangos de cada hiperparámetro a explorar para cada modelo.

In [9]:
# ==============================================================================
# 6. ESPACIO DE BÚSQUEDA DE HIPERPARÁMETROS
# ==============================================================================

SEARCH_SPACES = {
    "LSTM": {
        "lstm_1":        [8, 16, 32],
        "dropout":       [0.2, 0.3],
        "lstm_2":        [8, 16, 32],
        "learning_rate": [1e-3],
        "batch_size":    [512],
    },
    "CNN": {
        "filters_1":     [16, 32, 64],
        "kernel_1":      [3, 5],
        "filters_2":     [16, 32],
        "kernel_2":      [3],
        "dropout":       [0.2, 0.3],
        "learning_rate": [1e-3],
        "batch_size":    [512],
    },
    "BiGRU": {
        "gru_units":     [8, 16, 32],
        "dropout":       [0.2, 0.3],
        "dense_units":   [8, 16],
        "learning_rate": [1e-3],
        "batch_size":    [512],
    },
}

# Mostrar tamaño de cada espacio
for nombre, espacio in SEARCH_SPACES.items():
    total_combos = 1
    for v in espacio.values():
        total_combos *= len(v)
    print(f"📐 {nombre}: {total_combos:,} combinaciones posibles → se probarán {N_TRIALS}")

📐 LSTM: 18 combinaciones posibles → se probarán 10
📐 CNN: 24 combinaciones posibles → se probarán 10
📐 BiGRU: 12 combinaciones posibles → se probarán 10


### **7. FUNCIONES CONSTRUCTORAS DE MODELOS**

Cada función recibe un diccionario de hiperparámetros y construye + compila el modelo correspondiente. La arquitectura es **idéntica** a la del notebook principal.

In [10]:
# ==============================================================================
# 7. CONSTRUCTORES DE MODELOS PARAMETRIZADOS
# ==============================================================================

def build_lstm_model(hp):
    """Construye el modelo LSTM con los hiperparámetros dados."""
    inp = Input(shape=(WINDOW_SIZE, NUM_FEATURES), name="Input_LSTM")
    x = LSTM(hp['lstm_1'], return_sequences=True)(inp)
    x = BatchNormalization()(x)
    x = Dropout(hp['dropout'])(x)
    x = LSTM(hp['lstm_2'])(x)
    x = BatchNormalization()(x)
    
    out_class = Dense(NUM_CLASSES, activation='softmax', name="out_class")(x)
    out_reg   = Dense(1, activation='sigmoid', name="out_reg")(x)
    
    modelo = Model(inputs=inp, outputs=[out_class, out_reg], name="LSTM_Tuning")
    modelo.compile(
        optimizer=Adam(learning_rate=hp['learning_rate']),
        loss={"out_class": "sparse_categorical_crossentropy", "out_reg": "mse"},
        loss_weights={"out_class": 1.0, "out_reg": 1.0},
        metrics={"out_class": ["accuracy"], "out_reg": ["mae"]}
    )
    return modelo


def build_cnn_model(hp):
    """Construye el modelo CNN-1D con los hiperparámetros dados."""
    inp = Input(shape=(WINDOW_SIZE, NUM_FEATURES), name="Input_CNN")
    x = Conv1D(filters=hp['filters_1'], kernel_size=hp['kernel_1'],
               activation='relu', padding='same')(inp)
    x = BatchNormalization()(x)
    x = Conv1D(filters=hp['filters_2'], kernel_size=hp['kernel_2'],
               activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(hp['dropout'])(x)
    
    out_class = Dense(NUM_CLASSES, activation='softmax', name="out_class")(x)
    out_reg   = Dense(1, activation='sigmoid', name="out_reg")(x)
    
    modelo = Model(inputs=inp, outputs=[out_class, out_reg], name="CNN_Tuning")
    modelo.compile(
        optimizer=Adam(learning_rate=hp['learning_rate']),
        loss={"out_class": "sparse_categorical_crossentropy", "out_reg": "mse"},
        loss_weights={"out_class": 1.0, "out_reg": 1.0},
        metrics={"out_class": ["accuracy"], "out_reg": ["mae"]}
    )
    return modelo


def build_bigru_model(hp):
    """Construye el modelo BiGRU con los hiperparámetros dados."""
    inp = Input(shape=(WINDOW_SIZE, NUM_FEATURES), name="Input_BiGRU")
    x = Bidirectional(GRU(hp['gru_units'], return_sequences=False))(inp)
    x = BatchNormalization()(x)
    x = Dropout(hp['dropout'])(x)
    x = Dense(hp['dense_units'], activation='relu')(x)
    
    out_class = Dense(NUM_CLASSES, activation='softmax', name="out_class")(x)
    out_reg   = Dense(1, activation='sigmoid', name="out_reg")(x)
    
    modelo = Model(inputs=inp, outputs=[out_class, out_reg], name="BiGRU_Tuning")
    modelo.compile(
        optimizer=Adam(learning_rate=hp['learning_rate']),
        loss={"out_class": "sparse_categorical_crossentropy", "out_reg": "mse"},
        loss_weights={"out_class": 1.0, "out_reg": 1.0},
        metrics={"out_class": ["accuracy"], "out_reg": ["mae"]}
    )
    return modelo


# Diccionario de constructores
MODEL_BUILDERS = {
    "LSTM":  build_lstm_model,
    "CNN":   build_cnn_model,
    "BiGRU": build_bigru_model,
}

print("✅ Constructores de modelos definidos: ", list(MODEL_BUILDERS.keys()))

✅ Constructores de modelos definidos:  ['LSTM', 'CNN', 'BiGRU']


### **8. EJECUCIÓN: FASE 1 (RANDOM SEARCH) + FASE 2 (CROSS-VALIDATION DEL MEJOR)**

Lógica del proceso para optimizar tiempos:

```
FASE 1: Random Search en Partición Única
- Usar los splits explícitos de Train y Val definidos previamente
- Para cada modelo:
    - Muestrear N_TRIALS combinaciones
    - Entrenar con EarlyStopping en ese único split
    - Registrar métricas y guardar el mejor (top 1)

FASE 2: K-Fold CV de Comprobación
- Para cada modelo:
    - Tomar los hiperparámetros ganadores de la FASE 1
    - Hacer un 5-Fold CV (GroupKFold) sobre TODO el dataset
    - Asegurar que no hay sobreajuste verificando las métricas promedio
```

In [11]:
# ==============================================================================
# 8. RANDOM SEARCH + K-FOLD CROSS-VALIDATION (PREPARACIÓN)
# ==============================================================================

def sample_unique_trials(space, n_trials):
    """Muestrea n_trials combinaciones únicas."""
    keys = list(space.keys())
    seen = set()
    trials = []
    max_iter = n_trials * 100
    attempt = 0
    while len(trials) < n_trials and attempt < max_iter:
        hp = {k: random.choice(v) for k, v in space.items()}
        hp_key = tuple(sorted(hp.items()))
        if hp_key not in seen:
            seen.add(hp_key)
            trials.append(hp)
        attempt += 1
    return trials

def scale_data_2d_to_3d(X_tr, X_va):
    """Escala datos 3D: fit en train, transform en ambos."""
    n_tr, w, nf = X_tr.shape
    n_va = X_va.shape[0]
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr.reshape(-1, nf)).reshape(n_tr, w, nf).astype(np.float32)
    X_va_sc = scaler.transform(X_va.reshape(-1, nf)).reshape(n_va, w, nf).astype(np.float32)
    return X_tr_sc, X_va_sc

# --- CRONÓMETRO GLOBAL ---
TIEMPO_INICIO_GLOBAL = time.time()

# ------------------------------------------------------------------------------
# PREPARACIÓN DE LA PARTICIÓN ÚNICA PARA FASE 1
# ------------------------------------------------------------------------------
print("🔪 Usando conjuntos Train y Val fijos para la Fase 1 (Tuning Hold-out)...")

X_tune_train_raw = X_train
yc_tune_train    = y_train_c
yr_tune_train    = y_train_r

X_tune_val_raw   = X_val
yc_tune_val      = y_val_c
yr_tune_val      = y_val_r

# Borrar referencias originales para liberar RAM después
del X_train, X_val, y_train_c, y_val_c, y_train_r, y_val_r
gc.collect()

print(f"   Train samples: {len(X_tune_train_raw)}")
print(f"   Val samples:   {len(X_tune_val_raw)}")

# Escalar partición de tuning
X_tune_train_scaled, X_tune_val_scaled = scale_data_2d_to_3d(X_tune_train_raw, X_tune_val_raw)

# Liberar RAW de tuning de memoria
del X_tune_train_raw, X_tune_val_raw
gc.collect()

# Inicialización de variables para guardar resultados
all_results = []
best_hyperparams = {}

import os
import json

csv_path = TUNING_OUTPUT_DIR / "all_tuning_trials.csv"
json_path = TUNING_OUTPUT_DIR / "optimal_hyperparameters.json"

🔪 Usando conjuntos Train y Val fijos para la Fase 1 (Tuning Hold-out)...
   Train samples: 1029109
   Val samples:   209396


**Optimización y CV de la red LSTM**

In [ ]:
# ==============================================================================
# OPTIMIZACIÓN Y CV: LSTM (CON GUARDADO CONTINUO)
# ==============================================================================
model_name = 'LSTM'
tiempo_inicio_modelo = time.time()
trials = sample_unique_trials(SEARCH_SPACES[model_name], N_TRIALS)

best_score_tune = -np.inf
best_hp = None

for trial_idx, hp in enumerate(trials):
    trial_start = time.time()
    
    early_stop = EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING, restore_best_weights=True, verbose=0)
    model = MODEL_BUILDERS[model_name](hp)
    
    hist = model.fit(
        X_tune_train_scaled, {"out_class": yc_tune_train, "out_reg": yr_tune_train},
        validation_data=(X_tune_val_scaled, {"out_class": yc_tune_val, "out_reg": yr_tune_val}),
        epochs=EPOCHS_TUNE, batch_size=hp['batch_size'],
        callbacks=[early_stop], verbose=0
    )
    
    best_epoch_idx = np.argmin(hist.history['val_loss'])
    val_acc  = hist.history['val_out_class_accuracy'][best_epoch_idx]
    val_loss = hist.history['val_loss'][best_epoch_idx]
    val_mae  = hist.history['val_out_reg_mae'][best_epoch_idx]
    epochs_run = best_epoch_idx + 1
    trial_time = (time.time() - trial_start) / 60.0
    
    print(f"   --- Trial {trial_idx+1}/{len(trials)} | Acc: {val_acc:.4f} | Loss: {val_loss:.4f} | Ep: {epochs_run} | Tiempo: {trial_time:.1f}m")
    
    # --- 💾 GUARDADO CONTINUO A CSV DESPUÉS DE CADA TRIAL ---
    result_row = {'modelo': model_name, 'trial': trial_idx + 1, 'val_acc': round(val_acc, 6), 'val_loss': round(val_loss, 6), 'val_mae': round(val_mae, 6), 'epochs': epochs_run, 'trial_time_min': round(trial_time, 2)}
    result_row.update({f'hp_{k}': v for k, v in hp.items()})
    
    df_trial = pd.DataFrame([result_row])
    file_exists = os.path.isfile(csv_path)
    df_trial.to_csv(csv_path, mode='a', header=not file_exists, index=False)
    
    if val_acc > best_score_tune:
        best_score_tune = val_acc
        best_hp = hp.copy()
        best_hp['_tune_val_acc'] = val_acc
        best_hp['_tune_val_loss'] = val_loss
        
    del model
    tf.keras.backend.clear_session()
    gc.collect()

print(f"\n🏆 Mejor HP para {model_name} (Hold-out): Acc={best_score_tune:.4f} -> {best_hp}")

# FASE 2: CROSS-VALIDATION
print(f"\n FASE 2: K-FOLD CROSS-VALIDATION (K={N_FOLDS}) para el mejor modelo {model_name}")
gkf = GroupKFold(n_splits=N_FOLDS)
fold_scores_acc, fold_scores_loss, fold_scores_mae = [], [], []

fold_idx = 1
for train_idx_cv, val_idx_cv in gkf.split(X_all, y_all_c, groups=groups_all):
    print(f"      -> Entrenando Fold {fold_idx}/{N_FOLDS}...")
    X_tr_sc_cv, X_va_sc_cv = scale_data_2d_to_3d(X_all[train_idx_cv], X_all[val_idx_cv])
    
    model_cv = MODEL_BUILDERS[model_name](best_hp)
    early_stop_cv = EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING, restore_best_weights=True, verbose=0)
    
    hist_cv = model_cv.fit(
        X_tr_sc_cv, {"out_class": y_all_c[train_idx_cv], "out_reg": y_all_r[train_idx_cv]},
        validation_data=(X_va_sc_cv, {"out_class": y_all_c[val_idx_cv], "out_reg": y_all_r[val_idx_cv]}),
        epochs=EPOCHS_TUNE, batch_size=best_hp['batch_size'], callbacks=[early_stop_cv], verbose=0
    )
    
    best_ep_cv = np.argmin(hist_cv.history['val_loss'])
    fold_scores_acc.append(hist_cv.history['val_out_class_accuracy'][best_ep_cv])
    fold_scores_loss.append(hist_cv.history['val_loss'][best_ep_cv])
    fold_scores_mae.append(hist_cv.history['val_out_reg_mae'][best_ep_cv])
    
    del model_cv, X_tr_sc_cv, X_va_sc_cv
    tf.keras.backend.clear_session()
    gc.collect()
    fold_idx += 1

best_hp['_cv_mean_val_acc'] = np.mean(fold_scores_acc)
best_hp['_cv_std_val_acc']  = np.std(fold_scores_acc)
best_hp['_cv_mean_val_loss']= np.mean(fold_scores_loss)
best_hp['_cv_mean_val_mae'] = np.mean(fold_scores_mae)

# --- 💾 GUARDADO CONTINUO A JSON TRAS EL CROSS-VALIDATION ---
json_export = json.load(open(json_path, 'r', encoding='utf-8')) if os.path.exists(json_path) else {}
json_export[model_name] = {k: (int(v) if isinstance(v, np.integer) else float(v) if isinstance(v, np.floating) else v) for k, v in best_hp.items()}
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(json_export, f, indent=4, ensure_ascii=False)

print(f"   🎯 Resultados CV para {model_name} guardados: Acc={best_hp['_cv_mean_val_acc']:.4f}")

   --- Trial 1/10 | Acc: 0.7079 | Loss: 0.9187 | Ep: 3 | Tiempo: 26.7m

   --- Trial 2/10 | Acc: 0.7407 | Loss: 0.7325 | Ep: 5 | Tiempo: 35.1m
   --- Trial 3/10 | Acc: 0.6648 | Loss: 0.9504 | Ep: 1 | Tiempo: 15.5m
   --- Trial 4/10 | Acc: 0.7380 | Loss: 0.7616 | Ep: 4 | Tiempo: 16.2m
   --- Trial 5/10 | Acc: 0.6917 | Loss: 0.8116 | Ep: 7 | Tiempo: 17.1m
   --- Trial 6/10 | Acc: 0.7168 | Loss: 0.9557 | Ep: 3 | Tiempo: 32.2m
   --- Trial 7/10 | Acc: 0.6963 | Loss: 0.8796 | Ep: 1 | Tiempo: 37.2m
   --- Trial 8/10 | Acc: 0.6775 | Loss: 0.8876 | Ep: 1 | Tiempo: 23.1m
   --- Trial 9/10 | Acc: 0.7116 | Loss: 0.7847 | Ep: 4 | Tiempo: 14.0m
   --- Trial 10/10 | Acc: 0.7236 | Loss: 0.7484 | Ep: 2 | Tiempo: 20.1m

🏆 Mejor HP para LSTM (Hold-out): Acc=0.7407 -> {'lstm_1': 8, 'dropout': 0.2, 'lstm_2': 32, 'learning_rate': 0.001, 'batch_size': 512, '_tune_val_acc': 0.7406540513038635, '_tune_val_loss': 0.7324516773223877}

 FASE 2: K-FOLD CROSS-VALIDATION (K=5) para el mejor modelo LSTM
      -> Ent

: 

**Optimización y CV de la CNN:**

In [12]:
# ==============================================================================
# OPTIMIZACIÓN Y CV: CNN (CON GUARDADO CONTINUO)
# ==============================================================================
model_name = 'CNN'
tiempo_inicio_modelo = time.time()
trials = sample_unique_trials(SEARCH_SPACES[model_name], N_TRIALS)

best_score_tune = -np.inf
best_hp = None

for trial_idx, hp in enumerate(trials):
    trial_start = time.time()
    
    early_stop = EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING, restore_best_weights=True, verbose=0)
    model = MODEL_BUILDERS[model_name](hp)
    
    hist = model.fit(
        X_tune_train_scaled, {"out_class": yc_tune_train, "out_reg": yr_tune_train},
        validation_data=(X_tune_val_scaled, {"out_class": yc_tune_val, "out_reg": yr_tune_val}),
        epochs=EPOCHS_TUNE, batch_size=hp['batch_size'],
        callbacks=[early_stop], verbose=0
    )
    
    best_epoch_idx = np.argmin(hist.history['val_loss'])
    val_acc  = hist.history['val_out_class_accuracy'][best_epoch_idx]
    val_loss = hist.history['val_loss'][best_epoch_idx]
    val_mae  = hist.history['val_out_reg_mae'][best_epoch_idx]
    epochs_run = best_epoch_idx + 1
    trial_time = (time.time() - trial_start) / 60.0
    
    print(f"   --- Trial {trial_idx+1}/{len(trials)} | Acc: {val_acc:.4f} | Loss: {val_loss:.4f} | Ep: {epochs_run} | Tiempo: {trial_time:.1f}m")
    
    # --- 💾 GUARDADO CONTINUO A CSV DESPUÉS DE CADA TRIAL ---
    result_row = {'modelo': model_name, 'trial': trial_idx + 1, 'val_acc': round(val_acc, 6), 'val_loss': round(val_loss, 6), 'val_mae': round(val_mae, 6), 'epochs': epochs_run, 'trial_time_min': round(trial_time, 2)}
    result_row.update({f'hp_{k}': v for k, v in hp.items()})
    
    df_trial = pd.DataFrame([result_row])
    file_exists = os.path.isfile(csv_path)
    df_trial.to_csv(csv_path, mode='a', header=not file_exists, index=False)
    
    if val_acc > best_score_tune:
        best_score_tune = val_acc
        best_hp = hp.copy()
        best_hp['_tune_val_acc'] = val_acc
        best_hp['_tune_val_loss'] = val_loss
        
    del model
    tf.keras.backend.clear_session()
    gc.collect()

print(f"\n🏆 Mejor HP para {model_name} (Hold-out): Acc={best_score_tune:.4f} -> {best_hp}")

# FASE 2: CROSS-VALIDATION
print(f"\n FASE 2: K-FOLD CROSS-VALIDATION (K={N_FOLDS}) para el mejor modelo {model_name}")
gkf = GroupKFold(n_splits=N_FOLDS)
fold_scores_acc, fold_scores_loss, fold_scores_mae = [], [], []

fold_idx = 1
for train_idx_cv, val_idx_cv in gkf.split(X_all, y_all_c, groups=groups_all):
    print(f"      -> Entrenando Fold {fold_idx}/{N_FOLDS}...")
    X_tr_sc_cv, X_va_sc_cv = scale_data_2d_to_3d(X_all[train_idx_cv], X_all[val_idx_cv])
    
    model_cv = MODEL_BUILDERS[model_name](best_hp)
    early_stop_cv = EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING, restore_best_weights=True, verbose=0)
    
    hist_cv = model_cv.fit(
        X_tr_sc_cv, {"out_class": y_all_c[train_idx_cv], "out_reg": y_all_r[train_idx_cv]},
        validation_data=(X_va_sc_cv, {"out_class": y_all_c[val_idx_cv], "out_reg": y_all_r[val_idx_cv]}),
        epochs=EPOCHS_TUNE, batch_size=best_hp['batch_size'], callbacks=[early_stop_cv], verbose=0
    )
    
    best_ep_cv = np.argmin(hist_cv.history['val_loss'])
    fold_scores_acc.append(hist_cv.history['val_out_class_accuracy'][best_ep_cv])
    fold_scores_loss.append(hist_cv.history['val_loss'][best_ep_cv])
    fold_scores_mae.append(hist_cv.history['val_out_reg_mae'][best_ep_cv])
    
    del model_cv, X_tr_sc_cv, X_va_sc_cv
    tf.keras.backend.clear_session()
    gc.collect()
    fold_idx += 1

best_hp['_cv_mean_val_acc'] = np.mean(fold_scores_acc)
best_hp['_cv_std_val_acc']  = np.std(fold_scores_acc)
best_hp['_cv_mean_val_loss']= np.mean(fold_scores_loss)
best_hp['_cv_mean_val_mae'] = np.mean(fold_scores_mae)

# --- 💾 GUARDADO CONTINUO A JSON TRAS EL CROSS-VALIDATION ---
json_export = json.load(open(json_path, 'r', encoding='utf-8')) if os.path.exists(json_path) else {}
json_export[model_name] = {k: (int(v) if isinstance(v, np.integer) else float(v) if isinstance(v, np.floating) else v) for k, v in best_hp.items()}
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(json_export, f, indent=4, ensure_ascii=False)

print(f"   🎯 Resultados CV para {model_name} guardados: Acc={best_hp['_cv_mean_val_acc']:.4f}")

   --- Trial 1/10 | Acc: 0.6642 | Loss: 0.8905 | Ep: 2 | Tiempo: 2.6m

   --- Trial 2/10 | Acc: 0.7107 | Loss: 0.7538 | Ep: 9 | Tiempo: 2.6m
   --- Trial 3/10 | Acc: 0.6823 | Loss: 0.8956 | Ep: 5 | Tiempo: 4.6m
   --- Trial 4/10 | Acc: 0.7204 | Loss: 0.7500 | Ep: 4 | Tiempo: 2.3m
   --- Trial 5/10 | Acc: 0.7146 | Loss: 0.7738 | Ep: 4 | Tiempo: 3.2m
   --- Trial 6/10 | Acc: 0.6958 | Loss: 0.7721 | Ep: 7 | Tiempo: 3.5m
   --- Trial 7/10 | Acc: 0.6913 | Loss: 0.8089 | Ep: 7 | Tiempo: 3.6m
   --- Trial 8/10 | Acc: 0.7102 | Loss: 0.7784 | Ep: 3 | Tiempo: 2.6m
   --- Trial 9/10 | Acc: 0.6933 | Loss: 0.8385 | Ep: 10 | Tiempo: 3.9m
   --- Trial 10/10 | Acc: 0.6992 | Loss: 0.8186 | Ep: 4 | Tiempo: 2.4m

🏆 Mejor HP para CNN (Hold-out): Acc=0.7204 -> {'filters_1': 16, 'kernel_1': 3, 'filters_2': 32, 'kernel_2': 3, 'dropout': 0.2, 'learning_rate': 0.001, 'batch_size': 512, '_tune_val_acc': 0.720414936542511, '_tune_val_loss': 0.7500178813934326}

 FASE 2: K-FOLD CROSS-VALIDATION (K=5) para el mejo

**Optimización y CV de la BiGRU:**

In [13]:
# ==============================================================================
# OPTIMIZACIÓN Y CV: BiGRU (CON GUARDADO CONTINUO)
# ==============================================================================
model_name = 'BiGRU'
tiempo_inicio_modelo = time.time()
trials = sample_unique_trials(SEARCH_SPACES[model_name], N_TRIALS)

best_score_tune = -np.inf
best_hp = None

for trial_idx, hp in enumerate(trials):
    trial_start = time.time()
    
    early_stop = EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING, restore_best_weights=True, verbose=0)
    model = MODEL_BUILDERS[model_name](hp)
    
    hist = model.fit(
        X_tune_train_scaled, {"out_class": yc_tune_train, "out_reg": yr_tune_train},
        validation_data=(X_tune_val_scaled, {"out_class": yc_tune_val, "out_reg": yr_tune_val}),
        epochs=EPOCHS_TUNE, batch_size=hp['batch_size'],
        callbacks=[early_stop], verbose=0
    )
    
    best_epoch_idx = np.argmin(hist.history['val_loss'])
    val_acc  = hist.history['val_out_class_accuracy'][best_epoch_idx]
    val_loss = hist.history['val_loss'][best_epoch_idx]
    val_mae  = hist.history['val_out_reg_mae'][best_epoch_idx]
    epochs_run = best_epoch_idx + 1
    trial_time = (time.time() - trial_start) / 60.0
    
    print(f"   --- Trial {trial_idx+1}/{len(trials)} | Acc: {val_acc:.4f} | Loss: {val_loss:.4f} | Ep: {epochs_run} | Tiempo: {trial_time:.1f}m")
    
    # --- 💾 GUARDADO CONTINUO A CSV DESPUÉS DE CADA TRIAL ---
    result_row = {'modelo': model_name, 'trial': trial_idx + 1, 'val_acc': round(val_acc, 6), 'val_loss': round(val_loss, 6), 'val_mae': round(val_mae, 6), 'epochs': epochs_run, 'trial_time_min': round(trial_time, 2)}
    result_row.update({f'hp_{k}': v for k, v in hp.items()})
    
    df_trial = pd.DataFrame([result_row])
    file_exists = os.path.isfile(csv_path)
    df_trial.to_csv(csv_path, mode='a', header=not file_exists, index=False)
    
    if val_acc > best_score_tune:
        best_score_tune = val_acc
        best_hp = hp.copy()
        best_hp['_tune_val_acc'] = val_acc
        best_hp['_tune_val_loss'] = val_loss
        
    del model
    tf.keras.backend.clear_session()
    gc.collect()

print(f"\n🏆 Mejor HP para {model_name} (Hold-out): Acc={best_score_tune:.4f} -> {best_hp}")

# FASE 2: CROSS-VALIDATION
print(f"\n FASE 2: K-FOLD CROSS-VALIDATION (K={N_FOLDS}) para el mejor modelo {model_name}")
gkf = GroupKFold(n_splits=N_FOLDS)
fold_scores_acc, fold_scores_loss, fold_scores_mae = [], [], []

fold_idx = 1
for train_idx_cv, val_idx_cv in gkf.split(X_all, y_all_c, groups=groups_all):
    print(f"      -> Entrenando Fold {fold_idx}/{N_FOLDS}...")
    X_tr_sc_cv, X_va_sc_cv = scale_data_2d_to_3d(X_all[train_idx_cv], X_all[val_idx_cv])
    
    model_cv = MODEL_BUILDERS[model_name](best_hp)
    early_stop_cv = EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING, restore_best_weights=True, verbose=0)
    
    hist_cv = model_cv.fit(
        X_tr_sc_cv, {"out_class": y_all_c[train_idx_cv], "out_reg": y_all_r[train_idx_cv]},
        validation_data=(X_va_sc_cv, {"out_class": y_all_c[val_idx_cv], "out_reg": y_all_r[val_idx_cv]}),
        epochs=EPOCHS_TUNE, batch_size=best_hp['batch_size'], callbacks=[early_stop_cv], verbose=0
    )
    
    best_ep_cv = np.argmin(hist_cv.history['val_loss'])
    fold_scores_acc.append(hist_cv.history['val_out_class_accuracy'][best_ep_cv])
    fold_scores_loss.append(hist_cv.history['val_loss'][best_ep_cv])
    fold_scores_mae.append(hist_cv.history['val_out_reg_mae'][best_ep_cv])
    
    del model_cv, X_tr_sc_cv, X_va_sc_cv
    tf.keras.backend.clear_session()
    gc.collect()
    fold_idx += 1

best_hp['_cv_mean_val_acc'] = np.mean(fold_scores_acc)
best_hp['_cv_std_val_acc']  = np.std(fold_scores_acc)
best_hp['_cv_mean_val_loss']= np.mean(fold_scores_loss)
best_hp['_cv_mean_val_mae'] = np.mean(fold_scores_mae)

# --- 💾 GUARDADO CONTINUO A JSON TRAS EL CROSS-VALIDATION ---
json_export = json.load(open(json_path, 'r', encoding='utf-8')) if os.path.exists(json_path) else {}
json_export[model_name] = {k: (int(v) if isinstance(v, np.integer) else float(v) if isinstance(v, np.floating) else v) for k, v in best_hp.items()}
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(json_export, f, indent=4, ensure_ascii=False)

print(f"   🎯 Resultados CV para {model_name} guardados: Acc={best_hp['_cv_mean_val_acc']:.4f}")

   --- Trial 1/10 | Acc: 0.7086 | Loss: 0.7978 | Ep: 2 | Tiempo: 14.8m
   --- Trial 2/10 | Acc: 0.7086 | Loss: 0.7804 | Ep: 6 | Tiempo: 9.6m
   --- Trial 3/10 | Acc: 0.6904 | Loss: 0.8034 | Ep: 10 | Tiempo: 10.7m
   --- Trial 4/10 | Acc: 0.7132 | Loss: 0.9232 | Ep: 3 | Tiempo: 24.3m
   --- Trial 5/10 | Acc: 0.6897 | Loss: 0.7965 | Ep: 4 | Tiempo: 33.1m
   --- Trial 6/10 | Acc: 0.7198 | Loss: 0.7368 | Ep: 3 | Tiempo: 26.9m
   --- Trial 7/10 | Acc: 0.7227 | Loss: 0.8048 | Ep: 10 | Tiempo: 62.2m
   --- Trial 8/10 | Acc: 0.7227 | Loss: 0.7229 | Ep: 9 | Tiempo: 12.4m
   --- Trial 9/10 | Acc: 0.7032 | Loss: 0.7740 | Ep: 10 | Tiempo: 16.8m
   --- Trial 10/10 | Acc: 0.7163 | Loss: 0.8600 | Ep: 4 | Tiempo: 58.5m

🏆 Mejor HP para BiGRU (Hold-out): Acc=0.7227 -> {'gru_units': 16, 'dropout': 0.2, 'dense_units': 8, 'learning_rate': 0.001, 'batch_size': 512, '_tune_val_acc': 0.7226881384849548, '_tune_val_loss': 0.8047540783882141}

 FASE 2: K-FOLD CROSS-VALIDATION (K=5) para el mejor modelo BiGRU
 

**Limpieza y tiempo total**

In [ ]:
# --- LIMPIEZA FINAL ---
del X_tune_train_scaled, X_tune_val_scaled
gc.collect()

TIEMPO_TOTAL_MINUTOS = (time.time() - TIEMPO_INICIO_GLOBAL) / 60.0
TIEMPO_TOTAL_HORAS   = TIEMPO_TOTAL_MINUTOS / 60.0

print(f"\n{'='*70}")
print(f"⏱️ TIEMPO TOTAL DEL PROCESO: {TIEMPO_TOTAL_MINUTOS:.2f} minutos ({TIEMPO_TOTAL_HORAS:.2f} horas)")
print(f"{'='*70}")

### **Resultados Fine-Tuning**

Mejores hiperparámetros de cada modelo:

* **LSTM:** {'lstm_1': 8, 'dropout': 0.2, 'lstm_2': 32, 'learning_rate': 0.001, 'batch_size': 512}
* **CNN:** {'filters_1': 16, 'kernel_1': 3, 'filters_2': 32, 'kernel_2': 3, 'dropout': 0.2, 'learning_rate': 0.001, 'batch_size': 512}
* **BiGRU:** {'gru_units': 16, 'dropout': 0.2, 'dense_units': 8, 'learning_rate': 0.001, 'batch_size': 512}

### **Resultados Cross Validation**

* **LSTM:** Accuracy = 0.7205 +/- 0.0184
* **CNN:** Accuracy = 0.7288 +/- 0.0118
* **BiGRU:** Accuracy =  0.6994 +/- 0.0202